# 05 · Preprocessing for Clustering
## NHANES 2017–March 2020 Women's CKM Phenotyping Project

---

**Author:** Alexandra Velez, OB-GYN (Colombia) | Data Science  
**Input:** `data/processed/ckm_features.csv` — 1,603 women aged 20–44  
**Output:** `data/processed/clustering_ready.csv` — scaled feature matrix
ready for notebook 06  
**Last updated:** 2026

---

### A revision, not the original plan

The first version of this notebook clustered on 10 features: 6 CKM biomarkers
plus 4 reproductive variables (Age_Menarche, Parity, Pregnancy_Loss,
APO_Score). Notebook 07 then tested whether GDM and macrosomia history were
enriched in the resulting clusters — and found that they were. The problem
is that APO_Score is a direct encoding of GDM and macrosomia history, and it
was one of the ten inputs the clustering algorithm used to build those
clusters in the first place. Testing an outcome for enrichment in groups
that were partly built from that same outcome is circular — it doesn't tell
you whether the biomarkers predict adverse pregnancy history, only that the
algorithm found the reproductive variables it was handed.

This version fixes that. Clustering now runs on the 6 CKM biomarkers only.
Age_Menarche, Parity, Pregnancy_Loss, and APO_Score are still prepared in
this notebook, but as descriptive and exposure variables carried forward for
notebook 07 — never as clustering inputs. Whatever relationship the
resulting clusters have with GDM or macrosomia history in notebook 07 is now
a genuine post-hoc test, not an artifact of the clustering itself.

### Purpose of this notebook

This notebook prepares the final feature matrix for k-means clustering in
notebook 06, and separately prepares the reproductive variables that
notebook 07 will test against the resulting clusters. No new analytical
decisions are made here beyond the biomarker-only scope change — the
scaling and missingness-handling choices were justified in notebook 04.

The preprocessing pipeline follows this sequence:

1. **Define the clustering feature set** — 6 CKM biomarkers only
   (HbA1c, BMI, Mean_SBP, eGFR, HDL, Glucose)
2. **Drop collinear features** — Waist (r=0.95 with BMI) and Mean_DBP
   (r=0.79 with Mean_SBP) were already excluded from this set for the same
   reason documented in notebook 04 Section 5
3. **Code reproductive variables descriptively** — never-pregnant women
   receive Parity=0, Pregnancy_Loss=0, APO_Score=0 by clinical definition;
   residual item non-response in Parity and Pregnancy_Loss is
   median-imputed; Age_Menarche and APO_Score are left with their natural
   missingness and carried forward as-is, since imputing them would risk
   reintroducing the same kind of leakage into notebook 07's enrichment test
4. **Scale features** — RobustScaler applied to the 6 clustering features
5. **Export** — a clustering-ready matrix for notebook 06 and a full
   reproductive-variable reference for notebook 07

### Decisions carried forward from notebook 04

| Decision | Rationale | Section |
|---|---|---|
| Cluster on 6 CKM biomarkers only | Reproductive/exposure variables must not inform the clusters they're later tested against | §1 |
| Drop Waist | Collinear with BMI, r=0.95 | §3 |
| Drop Mean_DBP | Collinear with Mean_SBP, r=0.79 | §3 |
| Zero-impute never-pregnant | Clinical definition, not imputation | §4 |
| Median impute Parity, Pregnancy_Loss | Small residual item non-response; descriptive use only | §4 |
| No imputation for Age_Menarche or APO_Score | Both are tested in notebook 07 — imputing them from the clustering matrix (or from each other) would reintroduce circularity | §4 |
| Retain medicated women | Exclusion biases against highest-risk group | §4 |
| RobustScaler | Skewed distributions with meaningful clinical outliers | §5 |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import RobustScaler
import warnings

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_PATH  = Path('../data/processed/ckm_features.csv')
OUTPUT_PATH = Path('../data/processed/clustering_ready.csv')
FIG_DIR     = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)

# ── Clustering features — 6 CKM biomarkers only ─────────────────────────────
# Age_Menarche, Parity, Pregnancy_Loss, and APO_Score are NOT here.
# They are reproductive/exposure variables tested against the clusters in
# notebook 07 — including them as clustering inputs would make that test
# circular. Waist and Mean_DBP are excluded for collinearity (Section 3).
CLUSTERING_FEATURES = [
    'HbA1c',
    'BMI',
    'Mean_SBP',
    'eGFR',
    'HDL',
    'Glucose',
]

REPRODUCTIVE_FEATURES = [
    'Age_Menarche',
    'Parity',
    'Pregnancy_Loss',
    'APO_Score',
]

# ── Plot style ─────────────────────────────────────────────────────────────
# Okabe-Ito colorblind-safe palette — consistent with prior notebooks
COLORS = {
    'no_apo':     '#0072B2',
    'macs_only':  '#56B4E9',
    'gdm_only':   '#E69F00',
    'gdm_macs':   '#D55E00',
    'never_preg': '#999999',
    'highlight':  '#CC79A7',
}

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')
print(f'  Input:   {INPUT_PATH}')
print(f'  Output:  {OUTPUT_PATH}')
print(f'  Figures: {FIG_DIR}')
print(f'\nClustering features ({len(CLUSTERING_FEATURES)}):')
for f in CLUSTERING_FEATURES:
    print(f'  {f}')
print(f'\nReproductive features carried forward as descriptive/exposure variables ({len(REPRODUCTIVE_FEATURES)}):')
for f in REPRODUCTIVE_FEATURES:
    print(f'  {f}')

Setup complete.
  Input:   ../data/processed/ckm_features.csv
  Output:  ../data/processed/clustering_ready.csv
  Figures: ../figures

Clustering features (6):
  HbA1c
  BMI
  Mean_SBP
  eGFR
  HDL
  Glucose

Reproductive features carried forward as descriptive/exposure variables (4):
  Age_Menarche
  Parity
  Pregnancy_Loss
  APO_Score


---
## Section 2 · Load & Validate

In [2]:
# ── Load CKM feature matrix ────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH, index_col='SEQN')

assert df.shape == (1603, 59), \
    f'Unexpected shape {df.shape} — expected (1603, 59)'
assert df.index.name == 'SEQN', \
    'SEQN is not the index'
assert all(f in df.columns for f in CLUSTERING_FEATURES), \
    'Some clustering features missing from input file'
assert all(f in df.columns for f in REPRODUCTIVE_FEATURES), \
    'Some reproductive features missing from input file'

print(f'✓ CKM feature matrix loaded: {df.shape}')
print(f'  Age range: {df["RIDAGEYR"].min():.0f}–{df["RIDAGEYR"].max():.0f} years')
print(f'  Ever-pregnant: {(df["Ever_Pregnant"] == 1).sum():,} women')
print(f'  Never-pregnant: {(df["Ever_Pregnant"] == 0).sum():,} women')

# ── Confirm clustering features present and check baseline missingness ──────
print(f'\nBaseline missingness — 6 CKM biomarker clustering features:')
print(f'  {"Feature":<20} {"Valid":>6} {"NaN":>6} {"% Missing":>10}')
print(f'  {"─"*20} {"─"*6} {"─"*6} {"─"*10}')

for feat in CLUSTERING_FEATURES:
    n_valid   = df[feat].notna().sum()
    n_nan     = df[feat].isna().sum()
    pct_miss  = n_nan / len(df) * 100
    print(f'  {feat:<20} {n_valid:>6,} {n_nan:>6,} {pct_miss:>9.1f}%')

print(f'\n✓ All assertions passed — ready for preprocessing')

✓ CKM feature matrix loaded: (1603, 59)
  Age range: 20–44 years
  Ever-pregnant: 1,150 women
  Never-pregnant: 451 women

Baseline missingness — 6 CKM biomarker clustering features:
  Feature               Valid    NaN  % Missing
  ──────────────────── ────── ────── ──────────
  HbA1c                 1,522     81       5.1%
  BMI                   1,596      7       0.4%
  Mean_SBP              1,451    152       9.5%
  eGFR                  1,499    104       6.5%
  HDL                   1,503    100       6.2%
  Glucose               1,497    106       6.6%

✓ All assertions passed — ready for preprocessing


---
## Section 3 · Drop Collinear Features

Two CKM biomarkers are excluded from the clustering feature set based on the
collinearity analysis in notebook 04 Section 5: Waist (r=0.95 with BMI) and
Mean_DBP (r=0.79 with Mean_SBP). Both are retained in the full dataframe for
descriptive characterization in notebook 07 — only their exclusion from
`CLUSTERING_FEATURES` matters for the clustering itself. This decision is
unrelated to the reproductive-variable scope change in this revision and is
carried forward unchanged from the original analysis.

In [3]:
# ── Drop collinear features from clustering set ────────────────────────────
# Waist and Mean_DBP remain in df for descriptive use in notebook 07
# They are excluded only from CLUSTERING_FEATURES, which was defined
# in Section 1 without them — no action needed on df itself

# ── Confirm neither feature is in the clustering set ──────────────────────
assert 'Waist' not in CLUSTERING_FEATURES, \
    'Waist should not be in clustering features'
assert 'Mean_DBP' not in CLUSTERING_FEATURES, \
    'Mean_DBP should not be in clustering features'

# ── Confirm both are still available in df for descriptive use ────────────
assert 'Waist' in df.columns, \
    'Waist missing from dataframe'
assert 'Mean_DBP' in df.columns, \
    'Mean_DBP missing from dataframe'

print('✓ Collinear features excluded from clustering set')
print(f'  Waist    — retained in df, excluded from clustering')
print(f'  Mean_DBP — retained in df, excluded from clustering')
print(f'\n✓ Clustering feature set confirmed: {len(CLUSTERING_FEATURES)} features')
print(f'  {CLUSTERING_FEATURES}')

✓ Collinear features excluded from clustering set
  Waist    — retained in df, excluded from clustering
  Mean_DBP — retained in df, excluded from clustering

✓ Clustering feature set confirmed: 6 features
  ['HbA1c', 'BMI', 'Mean_SBP', 'eGFR', 'HDL', 'Glucose']


---
## Section 4 · Reproductive Variable Coding (Descriptive)

Age_Menarche, Parity, Pregnancy_Loss, and APO_Score are **not** clustering
inputs in this notebook. They are the exposure variables notebook 07 tests
against the clusters that Section 5 below builds from biomarkers alone. This
section prepares them for that test and for descriptive characterization —
nothing here feeds into `CLUSTERING_FEATURES`.

Missingness in these variables is predominantly structural, arising from
NHANES skip logic rather than data collection failure, as documented in
notebook 04 Section 6. The steps below follow this order:

1. **Zero-impute never-pregnant women** — Parity, Pregnancy_Loss, and
   APO_Score set to 0 for the 451 women who reported never having been
   pregnant. This is a clinical definition, not statistical imputation.
2. **Median impute residual Parity and Pregnancy_Loss** — small item
   non-response among ever-pregnant women (15 and 50 women respectively)
3. **Flag — not impute or exclude — unknown pregnancy status and unknown
   APO status.** 2 women have unknown pregnancy status and 89 ever-pregnant
   women have unknown APO status. In the original version of this notebook
   these women were dropped from the clustering sample. That's no longer
   necessary — they have nothing wrong with their CKM biomarkers, so they
   remain eligible for clustering in Section 5. They're flagged here only so
   notebook 07 can exclude them from the specific test that needs a known
   APO status.
4. **Leave Age_Menarche unimputed.** The previous version filled 78 missing
   values with KNN imputation run on the full 10-feature clustering matrix.
   That's a second, subtler version of the same circularity problem —
   Age_Menarche was later compared across clusters in notebook 07, so
   imputing it from the clustering features would have partly manufactured
   the pattern it was used to detect. It's carried forward with its natural
   missingness instead.

In [4]:
# ── Step 1 — Zero-impute never-pregnant women ──────────────────────────────
# Clinical definition: a woman who has never been pregnant has zero
# deliveries, zero pregnancy losses, and zero adverse pregnancy outcomes
# This is not statistical imputation — it applies domain knowledge
# df_pre carries the reproductive variables forward; it is NOT the
# clustering input matrix (that's built from df directly in Section 5)

df_pre = df.copy()   # preserve original df for reference

never_pregnant_mask = df_pre['Ever_Pregnant'] == 0

n_never = never_pregnant_mask.sum()

df_pre.loc[never_pregnant_mask, 'Parity']          = 0
df_pre.loc[never_pregnant_mask, 'Pregnancy_Loss']  = 0
df_pre.loc[never_pregnant_mask, 'APO_Score']       = 0

# ── Validate ───────────────────────────────────────────────────────────────
assert df_pre.loc[never_pregnant_mask, 'Parity'].isna().sum() == 0, \
    'NaN remains in Parity after zero-impute'
assert df_pre.loc[never_pregnant_mask, 'Pregnancy_Loss'].isna().sum() == 0, \
    'NaN remains in Pregnancy_Loss after zero-impute'
assert df_pre.loc[never_pregnant_mask, 'APO_Score'].isna().sum() == 0, \
    'NaN remains in APO_Score after zero-impute'

print(f'✓ Step 1 complete — zero-imputed {n_never:,} never-pregnant women')
print(f'  Parity          NaN remaining: '
      f'{df_pre["Parity"].isna().sum()}')
print(f'  Pregnancy_Loss  NaN remaining: '
      f'{df_pre["Pregnancy_Loss"].isna().sum()}')
print(f'  APO_Score       NaN remaining: '
      f'{df_pre["APO_Score"].isna().sum()}')

✓ Step 1 complete — zero-imputed 451 never-pregnant women
  Parity          NaN remaining: 17
  Pregnancy_Loss  NaN remaining: 52
  APO_Score       NaN remaining: 91


### Unknown pregnancy status — 2 women, kept in the sample

2 women have `Ever_Pregnant` missing, so the zero-impute logic above
couldn't apply to them (their Parity, Pregnancy_Loss, and APO_Score remain
NaN). In the original version of this notebook, these 2 women were dropped
from the analysis entirely. That was only necessary because those fields
were clustering inputs — now that they're descriptive only, there's no
reason to lose these 2 women from the CKM biomarker clustering in Section 5.
They're kept in `df_pre` and will simply show up as "unknown" wherever
reproductive history is reported.

In [5]:
# ── Unknown pregnancy status — 2 women, kept in sample ─────────────────────
# Cannot apply zero-impute or ever-pregnant logic to these women; their
# Parity, Pregnancy_Loss, and APO_Score remain NaN. Unlike the previous
# version of this notebook, they are NOT dropped — pregnancy status has no
# bearing on their eligibility for CKM-biomarker-only clustering.

ep_nan_mask = df_pre['Ever_Pregnant'].isna()

print(f'Women with unknown pregnancy status: {ep_nan_mask.sum()}')
print(f'  Parity NaN among them:         '
      f'{df_pre.loc[ep_nan_mask, "Parity"].isna().sum()}')
print(f'  Pregnancy_Loss NaN among them: '
      f'{df_pre.loc[ep_nan_mask, "Pregnancy_Loss"].isna().sum()}')
print(f'  APO_Score NaN among them:      '
      f'{df_pre.loc[ep_nan_mask, "APO_Score"].isna().sum()}')
print(f'\n✓ Sample size unchanged: {len(df_pre):,} women '
      f'(no rows dropped)')

Women with unknown pregnancy status: 2
  Parity NaN among them:         2
  Pregnancy_Loss NaN among them: 2
  APO_Score NaN among them:      2

✓ Sample size unchanged: 1,603 women (no rows dropped)


### Step 2 · Median imputation — Parity and Pregnancy_Loss

Among ever-pregnant women, 15 had missing Parity and 50 had missing
Pregnancy_Loss due to item non-response. Median imputation was applied
within the ever-pregnant subgroup only — never-pregnant women already
received zero values in Step 1 and were not affected. The median Parity
among ever-pregnant women was 2.0 deliveries and the median Pregnancy_Loss
was 0.0 — both clinically sensible values for this sample. This is purely
descriptive coding; neither variable is a clustering input.

In [6]:
# ── Step 2 — Median impute Parity and Pregnancy_Loss ──────────────────────
# 15 and 50 ever-pregnant women respectively with item non-response
# Median imputation within ever-pregnant women only
# Never-pregnant women already have 0 from Step 1 — not affected
# Descriptive coding only — Parity and Pregnancy_Loss are not clustering
# inputs, so this has no bearing on notebook 06

ever_pregnant_mask = df_pre['Ever_Pregnant'] == 1

for feat in ['Parity', 'Pregnancy_Loss']:
    median_val = df_pre.loc[ever_pregnant_mask, feat].median()
    n_imputed  = df_pre.loc[ever_pregnant_mask, feat].isna().sum()

    df_pre.loc[ever_pregnant_mask & df_pre[feat].isna(), feat] = median_val

    print(f'✓ {feat:<20} median = {median_val:.1f}  '
          f'({n_imputed} values imputed)')

print(f'\nParity NaN remaining:         {df_pre["Parity"].isna().sum()} '
      f'(the 2 unknown-pregnancy-status women)')
print(f'Pregnancy_Loss NaN remaining: {df_pre["Pregnancy_Loss"].isna().sum()} '
      f'(the 2 unknown-pregnancy-status women)')

✓ Parity               median = 2.0  (15 values imputed)
✓ Pregnancy_Loss       median = 0.0  (50 values imputed)

Parity NaN remaining:         2 (the 2 unknown-pregnancy-status women)
Pregnancy_Loss NaN remaining: 2 (the 2 unknown-pregnancy-status women)


### Step 3 · Flag APO_Score unknowns — not excluded

89 ever-pregnant women did not answer the gestational diabetes or
macrosomia questions, leaving their APO_Score unknown. These women cannot
be imputed — APO_Score encodes discrete clinical events that either
occurred or did not, and assigning a statistical estimate to unknown
pregnancy outcomes would be clinically indefensible.

In the original version of this notebook, these 89 women were excluded
from the primary analysis sample because APO_Score was a clustering input
and k-means requires complete cases on every input feature. That
constraint is gone — APO_Score isn't a clustering input anymore, so an
unknown APO status is no longer a reason to exclude a woman from the
biomarker clustering in Section 5. These 89 women are flagged here (not
dropped) so notebook 07 can exclude them from the specific enrichment test
that requires a known APO status, while keeping them in the clustering
sample if their CKM biomarkers are complete.

In [7]:
# ── Step 3 — Flag APO_Score unknowns ──────────────────────────────────────
# 89 ever-pregnant women did not answer GDM or macrosomia questions
# APO_Score encodes discrete clinical events that cannot be statistically
# estimated. Unlike the previous version, these women are NOT excluded from
# df_pre or from clustering — they are flagged for exclusion only from
# notebook 07's APO enrichment test, which requires a known APO status.

apo_unknown_mask = ever_pregnant_mask & df_pre['APO_Score'].isna()
n_unknown         = apo_unknown_mask.sum()

# Reference subset for notebook 07 — women with unknown APO status
df_apo_unknown = df_pre[apo_unknown_mask].copy()

print(f'✓ Step 3 complete — {n_unknown} women flagged with unknown APO status')
print(f'  Sample size unchanged: {len(df_pre):,} women (no rows dropped)')
print(f'  These women remain eligible for CKM-biomarker clustering in Section 5')
print(f'  They will be excluded only from the notebook 07 APO enrichment test')

# ── APO distribution, full sample (n=1,603) ────────────────────────────────
print(f'\nAPO distribution, full sample (n={len(df_pre):,}):')
print(f'  {"Group":<25} {"N":>6} {"% of sample":>12}')
print(f'  {"─"*25} {"─"*6} {"─"*12}')

never_preg_mask = df_pre['Ever_Pregnant'] == 0
n_never = never_preg_mask.sum()
print(f'  {"Never pregnant":<25} {n_never:>6,} {n_never/len(df_pre)*100:>11.1f}%')

apo_map = {
    0.0: 'No APOs',
    1.0: 'Macrosomia only',
    2.0: 'GDM only',
    3.0: 'GDM + Macrosomia',
}

for score, label in apo_map.items():
    n = (ever_pregnant_mask & (df_pre['APO_Score'] == score)).sum()
    print(f'  {label:<25} {n:>6,} {n/len(df_pre)*100:>11.1f}%')

print(f'  {"Unknown pregnancy status":<25} {ep_nan_mask.sum():>6,} '
      f'{ep_nan_mask.sum()/len(df_pre)*100:>11.1f}%')
print(f'  {"Unknown APO status":<25} {n_unknown:>6,} '
      f'{n_unknown/len(df_pre)*100:>11.1f}%')
print(f'  {"─"*25} {"─"*6} {"─"*12}')
print(f'  {"Total":<25} {len(df_pre):>6,} {"100.0%":>12}')

✓ Step 3 complete — 89 women flagged with unknown APO status
  Sample size unchanged: 1,603 women (no rows dropped)
  These women remain eligible for CKM-biomarker clustering in Section 5
  They will be excluded only from the notebook 07 APO enrichment test

APO distribution, full sample (n=1,603):
  Group                          N  % of sample
  ───────────────────────── ────── ────────────
  Never pregnant               451        28.1%
  No APOs                      813        50.7%
  Macrosomia only              114         7.1%
  GDM only                     104         6.5%
  GDM + Macrosomia              30         1.9%
  Unknown pregnancy status       2         0.1%
  Unknown APO status            89         5.6%
  ───────────────────────── ────── ────────────
  Total                      1,603       100.0%


The distribution above is mutually exclusive and clinically coherent:
451 never-pregnant women (28.1%), 813 with no APOs (50.7%), 248 with at
least one APO (15.5%), 2 with unknown pregnancy status (0.1%), and 89 with
unknown APO status (5.6%). This is the full reproductive picture for the
1,603-woman analytical sample — none of these categories determine
clustering eligibility anymore. Clustering eligibility in Section 5 depends
only on whether a woman has complete data on the 6 CKM biomarkers.

### Step 4 · Age_Menarche — left unimputed

78 women have missing Age_Menarche due to random item non-response. The
previous version of this notebook filled these with KNN imputation (k=5)
run across the full 10-feature clustering matrix — using BMI, blood
pressure, and the other clustering features to estimate each woman's likely
menarche age from her nearest neighbors.

That approach has a problem once Age_Menarche is treated as an exposure
variable rather than a clustering input. Notebook 07 compares Age_Menarche
across the clusters that Section 5 builds from the 6 CKM biomarkers. If the
78 imputed values were generated using those same biomarkers as the
neighbor space, the imputed values would be systematically pulled toward
whatever biomarker profile a woman already has — manufacturing part of the
very cluster-level difference the comparison is looking for. It's a subtler
version of the same circularity problem this whole revision is fixing, so
it gets the same treatment: no imputation. The 78 missing values are left
as NaN and excluded only from analyses that specifically use Age_Menarche.

In [8]:
# ── Step 4 — Age_Menarche: report missingness, no imputation ───────────────
# No KNN imputation this time — see markdown above for why. Age_Menarche
# is carried forward with its natural missingness. The one implausible
# observed value (see below) is still corrected, since that's a data
# quality fix, not an imputation.

n_missing = df_pre['Age_Menarche'].isna().sum()
print(f'Age_Menarche NaN (left unimputed): {n_missing}')

observed = df_pre['Age_Menarche'].dropna()
print(f'\nObserved Age_Menarche distribution (n={len(observed):,}):')
print(f'  Min:    {observed.min():.1f}')
print(f'  Max:    {observed.max():.1f}')
print(f'  Mean:   {observed.mean():.2f}')
print(f'  Median: {observed.median():.1f}')

implausible = observed[observed < 8]
print(f'\nValues below age 8 (clinically implausible): {len(implausible)}')
if len(implausible) > 0:
    print(f'  Values: {sorted(implausible.values)}')

# ── Clip implausible value ──────────────────────────────────────────────────
# 1 woman with observed Age_Menarche = 6.0 — not physiologically plausible
# Precocious puberty is defined as menarche before age 8
# Value clipped to 8.0 — the clinical floor for menarche age
# This is a data-quality correction to an observed value, not imputation
n_clipped = (df_pre['Age_Menarche'] < 8).sum()
df_pre.loc[df_pre['Age_Menarche'] < 8, 'Age_Menarche'] = 8.0

assert (df_pre['Age_Menarche'] < 8).sum() == 0, \
    'Values below 8 remain after clipping'

observed_after = df_pre['Age_Menarche'].dropna()
print(f'\n✓ Clipped {n_clipped} implausible value(s) to the 8.0-year floor')
print(f'  Final observed distribution (n={len(observed_after):,}):')
print(f'  Min:    {observed_after.min():.1f}')
print(f'  Median: {observed_after.median():.1f}')
print(f'  Mean:   {observed_after.mean():.2f}')
print(f'\n✓ Age_Menarche NaN remaining: {df_pre["Age_Menarche"].isna().sum()} '
      f'(carried forward, excluded only from Age_Menarche-specific analyses)')

Age_Menarche NaN (left unimputed): 78

Observed Age_Menarche distribution (n=1,525):
  Min:    6.0
  Max:    20.0
  Mean:   12.61
  Median: 12.0

Values below age 8 (clinically implausible): 1
  Values: [np.float64(6.0)]

✓ Clipped 1 implausible value(s) to the 8.0-year floor
  Final observed distribution (n=1,525):
  Min:    8.0
  Median: 12.0
  Mean:   12.62

✓ Age_Menarche NaN remaining: 78 (carried forward, excluded only from Age_Menarche-specific analyses)


### Reproductive variable coding — summary

Every woman in the 1,603-woman sample now has clinically coded Parity,
Pregnancy_Loss, and APO_Score, with the sole exception of the 2 women with
unknown pregnancy status and, for APO_Score specifically, the 89 women with
unknown APO status. Age_Menarche retains its natural missingness of 78
women (77 after the clip, since the clip corrected rather than removed a
value). None of this determines who enters the clustering sample in Section
5 — that sample is defined entirely by CKM biomarker completeness.

---
## Section 5 · Feature Scaling

K-means clustering uses Euclidean distance to assign observations to
clusters. Features on larger numeric scales contribute disproportionately
to distance calculations regardless of their clinical importance — without
scaling, eGFR (range ~60–145) and Glucose (range ~50–380) would dominate
cluster assignments while HDL (range ~20–90) would contribute comparatively
little. Scaling is therefore not optional; it is a prerequisite for
meaningful clustering.

### Why not z-score standardization?

Z-score standardization (StandardScaler) is the most commonly used scaling
method and the default choice in most machine learning workflows. It
transforms each feature to mean = 0 and standard deviation = 1 using the
formula (x − mean) / std. For normally distributed features without
extreme values, it performs well and produces easily interpretable results.

However, z-score standardization has a fundamental vulnerability: both the
mean and the standard deviation are sensitive to outliers. In a clinical
dataset like this one, extreme values are not errors — they are real
patients with real disease. A woman with HbA1c of 13.0% has uncontrolled
diabetes. A woman with BMI of 58 has severe obesity. A woman with Glucose
of 380 mg/dL is in metabolic crisis. These values are clinically meaningful
and must be retained in the analysis.

The problem is that when z-score scaling is applied, these extreme values
inflate the standard deviation of their respective features. A larger
standard deviation compresses the scaled values for the majority of women
into a narrower band, reducing their contribution to distance calculations
and giving disproportionate influence to the outliers. In effect, z-score
scaling in a skewed clinical distribution does the opposite of what scaling
is supposed to do — instead of equalizing feature contributions, it
amplifies the influence of the most extreme cases.

This is not a theoretical concern. The biomarker distributions examined in
notebook 04 confirmed right-skewed distributions with meaningful clinical
outliers across HbA1c, BMI, Glucose, and eGFR. Z-score scaling is therefore
not appropriate for this dataset.

### Robust scaling

RobustScaler transforms each feature using the median and interquartile
range (IQR): (x − median) / IQR. Because the median and IQR are order-based
statistics, they are inherently resistant to extreme values — an outlier at
either tail does not affect the median or IQR of a large distribution. The
scaling parameters therefore reflect the central tendency and spread of the
majority of the sample, not the extremes.

The scaling parameters themselves are also clinically meaningful. The
median and IQR of HbA1c, BMI, and Mean_SBP in this sample are interpretable
statistics that a clinician can evaluate directly — unlike the mean and
standard deviation, which are distorted by the skewed distributions
documented in notebook 04.

**Decision: RobustScaler is applied to the 6 CKM biomarker clustering
features, fit on the full 1,603-woman sample.** Fitting on the full sample
rather than a reproductive-filtered subset is itself a consequence of this
revision — there's no longer a "primary sample" gated by reproductive
completeness, since reproductive variables no longer determine clustering
eligibility.

Cluster stability under StandardScaler is examined as part of the
sensitivity analysis in notebook 06 to confirm that the choice of scaling
method does not materially alter the cluster structure.

In [9]:
# ── Feature scaling — RobustScaler ────────────────────────────────────────
# Applied to the 6 CKM biomarker clustering features, fit on the full
# 1,603-woman sample. NaN values are ignored when computing the median/IQR
# and pass through unchanged — k-means will operate on complete cases only
# (quantified below).

# ── Extract clustering features ────────────────────────────────────────────
X = df[CLUSTERING_FEATURES].copy()

print(f'Feature matrix before scaling:')
print(f'  Shape: {X.shape}')
print(f'  Complete cases: {X.dropna().shape[0]:,}')
print(f'  Cases with any NaN: {X.shape[0] - X.dropna().shape[0]:,}')

# ── Fit and transform ──────────────────────────────────────────────────────
scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(
    X_scaled,
    index=df.index,
    columns=CLUSTERING_FEATURES
)

# ── Scaling parameters — document for reproducibility ─────────────────────
print(f'\nScaling parameters (median / IQR):')
print(f'  {"Feature":<20} {"Median":>10} {"IQR":>10}')
print(f'  {"─"*20} {"─"*10} {"─"*10}')

for i, feat in enumerate(CLUSTERING_FEATURES):
    median = scaler.center_[i]
    iqr    = scaler.scale_[i]
    print(f'  {feat:<20} {median:>10.3f} {iqr:>10.3f}')

# ── Validate scaled distribution ───────────────────────────────────────────
print(f'\nScaled feature ranges (should be centered near 0):')
print(f'  {"Feature":<20} {"Min":>8} {"Median":>8} {"Max":>8}')
print(f'  {"─"*20} {"─"*8} {"─"*8} {"─"*8}')

for feat in CLUSTERING_FEATURES:
    vals = df_scaled[feat].dropna()
    print(f'  {feat:<20} {vals.min():>8.2f} {vals.median():>8.2f} '
          f'{vals.max():>8.2f}')

print(f'\n✓ RobustScaler applied to all {len(CLUSTERING_FEATURES)} features')

Feature matrix before scaling:
  Shape: (1603, 6)
  Complete cases: 1,358
  Cases with any NaN: 245

Scaling parameters (median / IQR):
  Feature                  Median        IQR
  ──────────────────── ────────── ──────────
  HbA1c                     5.300      0.400
  BMI                      28.700     11.400
  Mean_SBP                108.333     15.667
  eGFR                    113.701     20.625
  HDL                      54.000     20.000
  Glucose                  88.000     12.000

Scaled feature ranges (should be centered near 0):
  Feature                   Min   Median      Max
  ──────────────────── ──────── ──────── ────────
  HbA1c                   -3.00     0.00    23.75
  BMI                     -1.24     0.00     5.58
  Mean_SBP                -2.04     0.00     4.74
  eGFR                    -5.28     0.00     1.40
  HDL                     -1.55     0.00     5.25
  Glucose                 -3.00     0.00    24.08

✓ RobustScaler applied to all 6 features


### Scaling results

RobustScaler was applied to the 6 CKM biomarker clustering features using
the median and IQR computed on the full 1,603-woman sample. All scaled
features are centered at 0 as expected — confirming the scaler was applied
correctly.

**Scaling parameters are clinically interpretable:** median HbA1c of 5.3%
and median glucose of 88.0 mg/dL are both within normal range — consistent
with a predominantly healthy young female sample. Median BMI of 28.7 sits
at the overweight threshold, and median SBP of 108.3 mmHg reflects the
characteristically lower blood pressure of reproductive-age women.

**Extreme right tails in metabolic features:** HbA1c reaches a scaled
maximum of 23.75 and Glucose of 24.08 — reflecting women with severely
uncontrolled diabetes at the far right of the distribution. These are real
clinical values retained intentionally. Under z-score scaling, these
extreme values would have inflated the standard deviation and distorted
the scaling for the majority of the sample — the primary reason
RobustScaler was chosen.

**Effective clustering sample:** of the 1,603 women in the full sample, 245
have missing values on at least one CKM biomarker due to incomplete
examination module participation. K-means requires complete cases — the
impact on effective sample size is quantified in the next cell.

In [10]:
# ── Complete cases quantification ──────────────────────────────────────────
# K-means requires complete cases — quantify impact of CKM biomarker
# missingness on effective clustering sample

print('=== Complete Cases Analysis ===\n')

# ── Overall complete cases ─────────────────────────────────────────────────
n_total    = len(df_scaled)
n_complete = df_scaled.dropna().shape[0]
n_missing  = n_total - n_complete

print(f'  Full sample:             {n_total:,}')
print(f'  Complete cases:          {n_complete:,} ({n_complete/n_total*100:.1f}%)')
print(f'  Incomplete cases:        {n_missing:,} ({n_missing/n_total*100:.1f}%)')

# ── Missingness by feature ─────────────────────────────────────────────────
print(f'\n  Missingness by CKM biomarker:')
print(f'  {"Feature":<20} {"NaN":>6} {"% Missing":>10}')
print(f'  {"─"*20} {"─"*6} {"─"*10}')

for feat in sorted(CLUSTERING_FEATURES, key=lambda f: -df_scaled[f].isna().sum()):
    n_nan = df_scaled[feat].isna().sum()
    pct   = n_nan / n_total * 100
    print(f'  {feat:<20} {n_nan:>6,} {pct:>9.1f}%')

# ── APO distribution among complete cases vs. full sample ─────────────────
df_complete = df_scaled.dropna().copy()
df_complete['Ever_Pregnant'] = df_pre.loc[df_complete.index, 'Ever_Pregnant']
df_complete['APO_Score_raw'] = df_pre.loc[df_complete.index, 'APO_Score']

print(f'\n  APO distribution among complete cases (n={n_complete:,}) '
      f'vs. full sample (n={n_total:,}):')
print(f'  {"Group":<25} {"Complete":>10} {"Full sample":>12}')
print(f'  {"─"*25} {"─"*10} {"─"*12}')

never_preg_complete = (df_complete['Ever_Pregnant'] == 0).sum()
never_preg_full      = (df_pre['Ever_Pregnant'] == 0).sum()
print(f'  {"Never pregnant":<25} '
      f'{never_preg_complete/n_complete*100:>9.1f}% {never_preg_full/n_total*100:>11.1f}%')

apo_map = {
    0.0: 'No APOs',
    1.0: 'Macrosomia only',
    2.0: 'GDM only',
    3.0: 'GDM + Macrosomia',
}
ever_preg_complete = df_complete['Ever_Pregnant'] == 1
ever_preg_full      = df_pre['Ever_Pregnant'] == 1
for score, label in apo_map.items():
    n_c = (ever_preg_complete & (df_complete['APO_Score_raw'] == score)).sum()
    n_f = (ever_preg_full & (df_pre['APO_Score'] == score)).sum()
    print(f'  {label:<25} {n_c/n_complete*100:>9.1f}% {n_f/n_total*100:>11.1f}%')

# ── Assertion ──────────────────────────────────────────────────────────────
assert n_complete >= 1200, \
    f'Complete cases too low for reliable clustering: {n_complete}'

print(f'\n✓ Effective clustering sample: {n_complete:,} women')
print(f'  ({n_missing} excluded due to missing CKM biomarkers — '
      f'{n_missing/n_total*100:.1f}% of full sample)')

=== Complete Cases Analysis ===

  Full sample:             1,603
  Complete cases:          1,358 (84.7%)
  Incomplete cases:        245 (15.3%)

  Missingness by CKM biomarker:
  Feature                 NaN  % Missing
  ──────────────────── ────── ──────────
  Mean_SBP                152       9.5%
  Glucose                 106       6.6%
  eGFR                    104       6.5%
  HDL                     100       6.2%
  HbA1c                    81       5.1%
  BMI                       7       0.4%

  APO distribution among complete cases (n=1,358) vs. full sample (n=1,603):
  Group                       Complete  Full sample
  ───────────────────────── ────────── ────────────
  Never pregnant                 28.8%        28.1%
  No APOs                        50.1%        50.7%
  Macrosomia only                 7.1%         7.1%
  GDM only                        6.8%         6.5%
  GDM + Macrosomia                1.8%         1.9%

✓ Effective clustering sample: 1,358 women
  (245 

### Complete cases analysis

Of the 1,603 women in the full sample, 1,358 (84.7%) have complete data
across all 6 CKM biomarker clustering features and form the effective
clustering sample for notebook 06. The 245 incomplete cases (15.3%) are
missing at least one CKM biomarker due to incomplete examination module
participation — the most common source being Mean_SBP (152 women, 9.5%),
followed by Glucose (106, 6.6%), eGFR (104, 6.5%), HDL (100, 6.2%), and
HbA1c (81, 5.1%). BMI is nearly complete (7 missing, 0.4%).

**Biomarker missingness is randomly distributed across APO subgroups.**
The APO distribution among complete cases is virtually identical to the
full sample — never-pregnant women represent 28.8% of complete cases vs.
28.1% of the full sample, GDM-only women 6.8% vs. 6.5%, and GDM +
macrosomia 1.8% vs. 1.9%. The 15.3% reduction in sample size from
incomplete biomarker data doesn't introduce meaningful selection bias into
the clustering analysis, and — because this filtering is now driven purely
by biomarker completeness rather than reproductive-variable completeness —
there's no mechanism by which it could correlate with GDM or macrosomia
history in a way that would bias notebook 07's enrichment test.

The effective clustering sample is **n = 1,358 women**, representing 84.7%
of the original analytical sample of 1,603 women. This is larger than the
1,285-woman sample the previous version produced, because that version also
excluded women for unknown pregnancy status (2) and unknown APO status
(89) — exclusions that no longer apply once those variables are out of the
clustering feature set.

---
## Section 6 · Export

Two files are exported for downstream use:

- **clustering_ready.csv** — the scaled 6-feature CKM biomarker matrix for
  the 1,358 complete cases, with reproductive and other descriptive
  variables carried forward unscaled for notebook 07. This is the direct
  input to notebook 06.
- **apo_unknown_reference.csv** — the subset of the clustering sample (72
  of the 1,358 women) with unknown APO status, for the specific exclusion
  notebook 07's enrichment test needs to make. Unlike the previous version,
  this is a reference subset, not an excluded group — these women remain
  in `clustering_ready.csv` and participate in clustering normally.

In [11]:
# ── Prepare final clustering matrix ───────────────────────────────────────
# Complete cases only — 1,358 women, 6 scaled CKM biomarker features
df_cluster = df_scaled.dropna().copy()

# Carry forward descriptive and exposure variables from df_pre for
# notebook 07. Age_Menarche, Parity, Pregnancy_Loss, and APO_Score are here
# now — they moved from clustering inputs to carried-forward exposure
# variables in this revision. Has_GDM and Has_Macrosomia are the raw
# outcome flags notebook 07's chi-square test uses directly.
carry_forward = [
    'Age_Menarche', 'Parity', 'Pregnancy_Loss', 'APO_Score',
    'Has_GDM', 'Has_Macrosomia',
    'Ever_Pregnant', 'Nulliparous',
    'On_Metformin', 'On_Statin', 'On_Antihypertensive',
    'On_Insulin', 'On_Hormonal_Contraception', 'Any_Prescription',
    'RIDAGEYR', 'RIDRETH3', 'INDFMPIR', 'DMDEDUC2',
    'Waist', 'Mean_DBP', 'WTSAFPRP'
]

for col in carry_forward:
    if col in df_pre.columns:
        df_cluster[col] = df_pre.loc[df_cluster.index, col]

# ── Reference subset — unknown APO status within the clustering sample ────
# Not excluded from df_cluster — just a convenience reference for notebook
# 07's enrichment test, which needs to filter these women out specifically
apo_unknown_in_cluster = df_cluster['APO_Score'].isna() & (df_cluster['Ever_Pregnant'] == 1)
df_apo_unknown_ref = df_cluster[apo_unknown_in_cluster].copy()

# ── Assertions ─────────────────────────────────────────────────────────────
assert df_cluster.shape[0] == 1358, \
    f'Unexpected row count: {df_cluster.shape[0]}'
assert df_cluster[CLUSTERING_FEATURES].isna().sum().sum() == 0, \
    'NaN present in clustering features'
assert df_cluster.index.name == 'SEQN', \
    'SEQN not index'

print(f'✓ Clustering matrix validated')
print(f'  Shape: {df_cluster.shape}')
print(f'  Clustering features: {len(CLUSTERING_FEATURES)} (all complete)')
print(f'  Descriptive/exposure variables carried forward: '
      f'{len([c for c in carry_forward if c in df_cluster.columns])}')
print(f'  Women with unknown APO status within clustering sample: '
      f'{apo_unknown_in_cluster.sum()}')

# ── Export ──────────────────────────────────────────────────────────────────
OUTPUT_PATH          = Path('../data/processed/clustering_ready.csv')
APO_UNKNOWN_REF_PATH = Path('../data/processed/apo_unknown_reference.csv')

df_cluster.to_csv(OUTPUT_PATH)
print(f'\n✓ clustering_ready.csv exported')
print(f'  Path: {OUTPUT_PATH}')
print(f'  Shape: {df_cluster.shape}')

df_apo_unknown_ref.to_csv(APO_UNKNOWN_REF_PATH)
print(f'\n✓ apo_unknown_reference.csv exported')
print(f'  Path: {APO_UNKNOWN_REF_PATH}')
print(f'  Shape: {df_apo_unknown_ref.shape}')

PRE_SCALE_PATH = Path('../data/processed/clustering_prescale.csv')
df_pre.to_csv(PRE_SCALE_PATH)
print(f'\n✓ Pre-scale feature matrix exported: {PRE_SCALE_PATH}')
print(f'  Shape: {df_pre.shape}')

✓ Clustering matrix validated
  Shape: (1358, 27)
  Clustering features: 6 (all complete)
  Descriptive/exposure variables carried forward: 21
  Women with unknown APO status within clustering sample: 72

✓ clustering_ready.csv exported
  Path: ../data/processed/clustering_ready.csv
  Shape: (1358, 27)

✓ apo_unknown_reference.csv exported
  Path: ../data/processed/apo_unknown_reference.csv
  Shape: (72, 27)

✓ Pre-scale feature matrix exported: ../data/processed/clustering_prescale.csv
  Shape: (1603, 59)


### Export

Three files are exported from this notebook:

**clustering_ready.csv (1,358 × 27):**
The primary output of this notebook — 1,358 complete cases with 6
RobustScaler-scaled CKM biomarker clustering features and 21 descriptive
and exposure variables carried forward in their original unscaled form.
This is the direct input to notebook 06. Age_Menarche, Parity,
Pregnancy_Loss, APO_Score, Has_GDM, and Has_Macrosomia are here as
exposure variables for notebook 07's enrichment test — not as clustering
inputs.

**apo_unknown_reference.csv (72 × 27):**
The 72 women within the clustering sample who have unknown APO status.
This is a reference subset for notebook 07 — it identifies which rows in
`clustering_ready.csv` to exclude from the APO enrichment test
specifically, without excluding those women from the clustering itself.
This is 72, not 89, because 17 of the 89 originally-flagged women were
already excluded from the clustering sample for having incomplete CKM
biomarker data.

**clustering_prescale.csv (1,603 × 59):**
The full pre-scaling feature matrix with reproductive variables coded per
Section 4, for any descriptive work in notebook 07 that needs the full
sample rather than the clustering subset.

---
## Section 7 · Notebook Summary & Handoff to Notebook 06

This notebook built the 6-feature CKM biomarker matrix that notebook 06
will cluster on, and separately prepared the reproductive variables that
notebook 07 will test against the resulting clusters — keeping the two
tracks independent is the entire point of this revision.

### Clustering sample — biomarker completeness only

| Step | Action | N removed | N remaining |
|---|---|---|---|
| Start | Full analytical sample | — | 1,603 |
| Section 5 | Incomplete CKM biomarker data | 245 | 1,358 |
| **Final** | **Effective clustering sample** | **—** | **1,358** |

This is a shorter cascade than the previous version's, and deliberately so.
The old cascade also removed 2 women for unknown pregnancy status and 89
for unknown APO status — both were only necessary because those variables
were clustering inputs. Now they aren't, and 91 fewer women are lost
before clustering even starts.

### Reproductive variable coding applied (descriptive only — not clustering inputs)

| Decision | Method | N affected |
|---|---|---|
| Never-pregnant zero-impute | Clinical definition | 451 |
| Parity residual imputation | Median (ever-pregnant) | 15 |
| Pregnancy_Loss residual imputation | Median (ever-pregnant) | 50 |
| Age_Menarche | Left unimputed — no KNN this time | 78 NaN carried forward |
| Age_Menarche floor clip | 8.0 years (precocious puberty threshold) | 1 |
| APO_Score | Left unknown for 89 women — flagged, not excluded | 89 flagged (72 within clustering sample) |
| Feature scaling | RobustScaler, full sample | All 6 clustering features |

### Final clustering feature set (n=1,358, 6 features)

| Feature | Median | IQR |
|---|---|---|
| HbA1c | 5.3% | 0.4 |
| BMI | 28.7 kg/m² | 11.4 |
| Mean_SBP | 108.3 mmHg | 15.7 |
| eGFR | 113.7 mL/min | 20.6 |
| HDL | 54.0 mg/dL | 20.0 |
| Glucose | 88.0 mg/dL | 12.0 |

### What notebook 06 will do

1. Load `clustering_ready.csv` — 1,358 women, 6 scaled biomarker features
2. Run k-means clustering across k=2–8
3. Evaluate solutions using the elbow method, silhouette score,
   Calinski-Harabasz index, and gap statistic
4. Run hierarchical clustering as a sensitivity check
5. Select optimal k based on internal metrics and clinical
   interpretability
6. Run sensitivity analyses: bootstrap stability and StandardScaler vs.
   RobustScaler comparison
7. Export cluster assignments for the non-circular enrichment test in
   notebook 07

Notebook 06's Section 8 in the previous version — an APO_Score=0 sensitivity
check for the 89 excluded women — is no longer needed. Those 89 women
were never excluded from clustering in this version, so there's nothing
to test.